# *This notebook provides a workflow to determine the best hyperparameters to build a SVM model before running a classification using them. Three hyperparameters are optimized*:
- *c*
- *gamma*
- *kernel*

The results from the final model can then be compared with the results of the Personalized PageRank (PPR) and Random Forest (RF) models in the context of a benchmark for a publication.

### *Importing the required libraries*

In [3]:
import sys
sys.path.append("../scripts")

import networkx as nx
import numpy as np
import os
import pandas as pd
import useful_functions
import shutil
import svm_benchmark
from itertools import product
from sklearn import svm
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, KFold, train_test_split
from sklearn.svm import SVC

### *Reading the training genes and setting the input parameters*

In [ ]:
# Setting the input dataset
dataset = "STRING_CS_100_E-GEOD-45750_corr_07"

# Setting the process of interest
process = "stalk_cell"

# Setting the type of parameters search wanted (Grid or Randomized)
search = "Randomized"

# Setting the output folder
dir_SVM = f"../results/{process}/{dataset}"
if not os.path.exists(dir_SVM):
    os.mkdir(dir_SVM)

dir_SVM = f"../results/{process}/{dataset}/SVM"
if not os.path.exists(dir_SVM):
    os.mkdir(dir_SVM)
else:
    shutil.rmtree(dir_SVM)
    os.mkdir(dir_SVM)

dir_benchmark = f"{dir_SVM}/Benchmark_hyperparameters"
if not os.path.exists(dir_benchmark):
    os.mkdir(dir_benchmark)
else:
    shutil.rmtree(dir_benchmark)
    os.mkdir(dir_benchmark)
    
# Reading the training genes
genes = pd.read_csv("../TrainingGenes/training_genes_stalk_cell.csv")
genes = genes["Feature"].to_list()

# Reading the input graph
graph = nx.read_graphml(f"../graphs/integrated/graph_{dataset}.graphml")

# Turning it into a matrix which will be used as input data for RF
data_graph = useful_functions.graph_to_matrix(graph)
print(f"Graph converted to a matrix with shape: {data_graph.shape}")

# Checking if the graph contains the training genes
valid_genes = [gene for gene in genes if gene in graph.nodes]
print(f"Number of training genes in the graph: {len(valid_genes)}")

### *Running a benchmark to determine the best hyperparameters*

In [ ]:
# Splitting the data into train (80%) and test (20%) sets
negative_genes = [gene for gene in data_graph.index if gene not in valid_genes]

train_size = int(0.8 * len(valid_genes))
train_genes_positive = valid_genes[:train_size]
test_genes_positive = valid_genes[train_size:]

train_size = int(0.8 * len(negative_genes))
train_genes_negative = negative_genes[:train_size]
test_genes_negative = negative_genes[train_size:]

train_genes = train_genes_positive + train_genes_negative
test_genes = test_genes_positive + test_genes_negative

labels_train = np.array([1 if gene in valid_genes else 0 for gene in train_genes])
labels_test = np.array([1 if gene in valid_genes else 0 for gene in test_genes])

data_graph_train = data_graph.loc[train_genes, :]
data_graph_test = data_graph.loc[test_genes, :]

# Training a SVM model without Hyperparameter tuning
clf = svm.SVC(kernel = "linear", probability = True)
clf.fit(data_graph_train, labels_train)
prediction = clf.predict_proba(data_graph_test)[:, 1]
accuracy, roc_auc, pr_auc = useful_functions.compute_metrics(prediction, labels_test)

performance_metrics = []
performance_metrics.append({"accuracy": accuracy,
                           "roc_auc": roc_auc,
                           "pr_auc": pr_auc})

performance_df = pd.DataFrame(performance_metrics)
performance_file = f"{dir_benchmark}/svm_performances_entire_dataset_no_optimization.csv"
performance_df.to_csv(performance_file, sep = ",", index = False)

# Hyperparameters tuning 
if search == "Grid":
    # using GridSearchCV()
    
    param_grid = {'C': [0.1, 1, 10, 100, 1000], 
                  'gamma': [1, 0.1, 0.01, 0.001, 0.0001], 
                  'kernel': ['rbf', 'poly', 'linear', 'sigmoid']}

    grid = GridSearchCV(SVC(), 
                        param_grid, 
                        refit = True, 
                        verbose = 10, 
                        n_jobs = -1,
                        scoring = "accuracy") 
 
    grid.fit(data_graph_train, labels_train)
    best_hyperparameters = pd.DataFrame({"C": [grid.best_params_["C"]],
                                         "gamma": [grid.best_params_["gamma"]],
                                         "kernel": [grid.best_params_["kernel"]]})

    best_hyperparameters.to_csv(f"{dir_benchmark}/Best_hyperparameters.csv",
                                sep = ",", index = False)

    with open(f"{dir_benchmark}/Best_hyperparameters.txt", "w") as f_out:
        f_out.write(str(grid.best_estimator_))

    # Evaluating the optimized model
    grid_predictions = grid.predict(data_graph_test) 
    print(classification_report(labels_test, grid_predictions))

    class_report = classification_report(labels_test, grid_predictions)
    with open(f"{dir_benchmark}/Classification_report_optimized_model.txt", "w") as f_out:
        f_out.write(class_report)

elif search == "Randomized":

    # ALTERNATIVE : A quicker and but less accurate option is a RandomizedSearchCV
    # which tests a specific number of combinations (default = 10) and determines
    # the best hyperparameters found within this restricted search space

    param_grid = {'C': [0.1, 1, 10, 100, 1000], 
                  'gamma': [1, 0.1, 0.01, 0.001, 0.0001], 
                  'kernel': ['rbf', 'poly', 'linear', 'sigmoid']} 

    # Retrieving the number of all possible combinations
    all_combinations = list(product(param_grid["C"],
                                   param_grid["gamma"],
                                   param_grid["kernel"]))

    # For computational time reasons, we will test only 1/3 of all possible combinations
    grid = RandomizedSearchCV(SVC(), 
                              param_grid, 
                              refit = True, 
                              verbose = 3,
                             n_iter = round(len(all_combinations) * 0.3))
 
    grid.fit(data_graph_train, labels_train)
    best_hyperparameters = pd.DataFrame({"C": [grid.best_params_["C"]],
                                         "gamma": [grid.best_params_["gamma"]],
                                         "kernel": [grid.best_params_["kernel"]]})

    best_hyperparameters.to_csv(f"{dir_benchmark}/Best_hyperparameters_RGS.csv",
                                sep = ",", index = False)

    with open(f"{dir_benchmark}/Best_hyperparameters_RGS.txt", "w") as f_out:
        f_out.write(str(grid.best_estimator_))

    # Evaluating the optimized model
    grid_predictions = grid.predict(data_graph_test) 
    print(classification_report(labels_test, grid_predictions))

    class_report = classification_report(labels_test, grid_predictions)
    with open(f"{dir_benchmark}/Classification_report_optimized_model_RGS.txt", "w") as f_out:
        f_out.write(class_report)

### *Running a SVM classification with the best hyperparameters*

In [ ]:
# Reading the best hyperparameters file
df_hyperparameters = pd.read_csv(f"{dir_benchmark}/Best_hyperparameters*.csv")

for parameter in df_hyperparameters.itertuples():
    best_C = parameter[1]
    best_gamma = parameter[2]
    best_kernel = parameter[3]

# Running a SVM model with the best hyperparameters on the whole dataset
svm_benchmark.svm_analysis_entire(data_graph,
                                 valid_genes,
                                 dir_benchmark,
                                 best_C,
                                 best_gamma,
                                 best_kernel)